In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from typing import Tuple, Dict
import os
import pydicom
from skimage.metrics import structural_similarity as ssim 
from skimage.filters import threshold_otsu


In [2]:
# =========================================================
# Config（所有可调参数都放在这里）
# =========================================================
# 路径与输出
BASE_DIR = Path.cwd()

OUTDIR = BASE_DIR / "output" / "limited_angle_proj_test"
os.makedirs(OUTDIR, exist_ok=True)

# 成对对比输出子文件夹（差值热图、指标表）
COMPARE_DIR = OUTDIR / "pairwise_compare"
os.makedirs(COMPARE_DIR, exist_ok=True)

# ---- 数据读取相关 ----
# 指定一个 DICOM 文件（单张切片）。如需从同一 study 里换切片，可把路径指向目标文件。
DCM_PATH = BASE_DIR / "data/raw/chest/manifest-1600709154662/LIDC-IDRI/LIDC-IDRI-0001/01-01-2000-NA-NA-30178/3000566.000000-NA-03192/1-090.dcm"

# 选用何种图作为底图：
# - "HU":     使用 DICOM 转换后的 HU 值（Hounsfield Unit）
# - "MU":     将 HU 转为线衰减系数 mu(mm^-1) 后作为投影底图（通常用于物理一致的投影）
BASE_IMAGE_TYPE = "MU"  # "HU" 或 "MU"

# HU->mu 转换参数（mu = MU_WATER * (1 + HU/1000)）
MU_WATER = 0.02  # mm^-1

# ---- Poisson 噪声设置（投影阶段）----
ADD_POISSON_NOISE = True      # 是否在投影阶段加入 Poisson 噪声
I0_PHOTONS = 1e5              # 每条射线的入射光子数 I0（可按曝光强度调整）
RNG_SEED = 12345              # 随机种子（复现实验）；设为 None 则不固定
POISSON_CLIP_MIN = 1.0        # 抽样后 Y 的最小值（避免 log(0)，通常取 1）

# 像素间距（优先使用 DICOM 字段 PixelSpacing；若缺失，则用此回退值）
PIXEL_MM_FALLBACK = 0.8  # mm

# ---- 投影/重建几何参数（扇束）----
# DSO: Source-to-Isocenter, DSD: Source-to-Detector
DSO = 600.0   # mm
DSD = 1000.0  # mm
DET_PITCH = 1.0      # 检测器通道间距(mm)
N_DET = 700          # 检测器通道数
U0 = 0.0             # 检测器中心偏移（通道）

# 角度步长列表（度）：将按每个步长分别采样生成 0–180、0–90、90–180 三组
# 例： [1, 2, 3, 5, 10]；你可按需调整
ANGLE_STEP_LIST = [1.0, 2.0, 3.0, 4.0, 5.0]

# 0–180 / 0–90 / 90–180 是否包含端点（题设要求：包含起止角）
INCLUDE_ENDPOINTS = True

# 重建模式与滤波器
MODE = "rebin"                 # 当前脚本只实现 rebin + 平行束 FBP
FILTER_NAME = "hann"           # "hann" | "ram-lak" | "shepp-logan"
HANN_CUTOFF = 0.8              # Hann 窗截止（0~1，1=Nyquist），仅当 FILTER_NAME="hann" 时生效
PARALLEL_N_T = 512             # 平行域 t 方向采样点数

# [SSIM] 计算相关（中心圆 vs Otsu ROI 二选一）
SSIM_USE_CENTER_CIRCLE = True   # True: 圆形中心 [ROI]；False: Otsu 阈值框 [ROI]
CENTER_CIRCLE_RADIUS_RATIO = 1  # 圆半径 = min(H,W)/2 * 该比例

# 可视化
PANEL_FIGSIZE = (12, 3)         # 四联图尺寸（宽,高）
SAVE_PER_RECON = True           # 是否保存每次重建的单图
SAVE_PANEL = True               # 是否保存四联图
SAVE_CSV = True                 # 是否保存统计 CSV

# 随机/数值稳定
EPS = 1e-12




In [3]:
# =========================================================
# 工具函数
# =========================================================
def hu_to_mu(hu: np.ndarray, mu_water: float = MU_WATER) -> np.ndarray:
    """[HU] -> mu(mm^-1)."""
    return mu_water * (1.0 + hu / 1000.0)


def bilinear_sample(img: np.ndarray, x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """双线性插值：在图像索引坐标 (x:col, y:row) 连续采样。"""
    H, W = img.shape
    x0 = np.floor(x).astype(int)
    y0 = np.floor(y).astype(int)
    x1 = x0 + 1
    y1 = y0 + 1

    x0 = np.clip(x0, 0, W-1); x1 = np.clip(x1, 0, W-1)
    y0 = np.clip(y0, 0, H-1); y1 = np.clip(y1, 0, H-1)

    Ia = img[y0, x0]
    Ib = img[y0, x1]
    Ic = img[y1, x0]
    Id = img[y1, x1]

    wa = (x1 - x) * (y1 - y)
    wb = (x - x0) * (y1 - y)
    wc = (x1 - x) * (y - y0)
    wd = (x - x0) * (y - y0)
    return Ia*wa + Ib*wb + Ic*wc + Id*wd


def ram_lak_filter(n: int, du: float, hann_cutoff: float | None = None) -> np.ndarray:
    """1D Ram-Lak(斜坡)滤波器（频域）。可选 Hann 窗。"""
    freqs = np.fft.fftfreq(n, d=du)  # cycles/mm
    filt = np.abs(freqs) * 2.0
    if hann_cutoff is not None:
        f_nyq = np.max(np.abs(freqs))
        cutoff = hann_cutoff * f_nyq
        w = 0.5 * (1 + np.cos(np.pi * np.clip(np.abs(freqs)/cutoff, 0, 1)))
        w[np.abs(freqs) > cutoff] = 0.0
        filt = filt * w
    return filt


def to_float01(a: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32)
    amin, amax = float(a.min()), float(a.max())
    if amax > amin:
        return (a - amin) / (amax - amin)
    return np.zeros_like(a, dtype=np.float32)


def to_uint8(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    imin, imax = img.min(), img.max()
    if imax > imin:
        img = (img - imin) / (imax - imin) * 255.0
    else:
        img = np.zeros_like(img)
    return img.astype(np.uint8)


def make_center_circle_mask(h: int, w: int, cy: float, cx: float, r: float) -> np.ndarray:
    yy, xx = np.ogrid[:h, :w]
    return (yy - cy)**2 + (xx - cx)**2 <= r**2


def ssim_circle(gt_img: np.ndarray, rec_img: np.ndarray,
                center: tuple[float, float] | None = None,
                radius: float | None = None,
                data_range: float = 1.0) -> float:
    """在中心圆形区域内计算 [SSIM]（通过对 SSIM map 做掩膜平均）。"""
    assert gt_img.shape == rec_img.shape, f"shape mismatch: {gt_img.shape} vs {rec_img.shape}"
    h, w = gt_img.shape[:2]
    if center is None:
        cy, cx = (h - 1) / 2.0, (w - 1) / 2.0
    else:
        cy, cx = float(center[0]), float(center[1])
    if radius is None:
        radius = CENTER_CIRCLE_RADIUS_RATIO * min(h, w) / 2.0
    r = float(radius)

    score, ssim_map = ssim(gt_img, rec_img, data_range=data_range, full=True)
    mask = make_center_circle_mask(h, w, cy, cx, r)
    masked_score = float(ssim_map[mask].mean())
    return masked_score


def ssim_roi_box(gt_img: np.ndarray, rec_img: np.ndarray) -> float:
    """使用 Otsu 对 GT 做阈值，取其外接矩形 [ROI]，在 [ROI] 内计算 [SSIM]。"""
    gt_n = to_float01(gt_img)
    thr = threshold_otsu(gt_n)
    roi = gt_n > thr
    ys, xs = np.where(roi)
    y0, y1 = ys.min(), ys.max()+1
    x0, x1 = xs.min(), xs.max()+1
    return float(ssim(gt_n[y0:y1, x0:x1], to_float01(rec_img)[y0:y1, x0:x1], data_range=1.0))


def rot90_cw(a: np.ndarray) -> np.ndarray:
    return np.rot90(a, k=-1)  # clockwise 90°

In [4]:
# =========================================================
# 核心：投影/重建
# =========================================================
def fanbeam_forward(mu_img: np.ndarray,
                    beta_deg: np.ndarray,
                    n_det: int = N_DET,
                    det_pitch_mm: float = DET_PITCH,
                    dso_mm: float = DSO,
                    dsd_mm: float = DSD,
                    pixel_mm: float = 0.8,
                    u0_offset: float = U0,
                    n_samples: int = 400) -> Tuple[np.ndarray, Dict]:
    """扇束前向投影（射线驱动）。返回：(fan-sinogram, info)
       若 ADD_POISSON_NOISE=True，则对光子计数做 Poisson 抽样并还原为线积分。
    """
    # 随机数生成器
    rng = np.random.default_rng(RNG_SEED) if RNG_SEED is not None else np.random.default_rng()

    H, W = mu_img.shape
    cx = (W - 1) / 2.0
    cy = (H - 1) / 2.0

    u = ((np.arange(n_det) - (n_det-1)/2.0 + u0_offset) * det_pitch_mm).astype(np.float32)
    P = np.zeros((len(beta_deg), n_det), dtype=np.float32)

    DID = dsd_mm - dso_mm
    t_vals = np.linspace(0.0, 1.0, n_samples).astype(np.float32)

    for ib, beta in enumerate(np.deg2rad(beta_deg)):
        xs = -dso_mm * np.cos(beta)
        ys = -dso_mm * np.sin(beta)

        xc = DID * np.cos(beta)
        yc = DID * np.sin(beta)
        tx = -np.sin(beta)
        ty =  np.cos(beta)

        xd = xc + u * tx
        yd = yc + u * ty

        dx = (xd - xs)[None, :]
        dy = (yd - ys)[None, :]

        X = xs + t_vals[:, None] * dx
        Y = ys + t_vals[:, None] * dy

        x_img = X / pixel_mm + cx
        y_img = Y / pixel_mm + cy

        mu_samp = bilinear_sample(mu_img, x_img, y_img)

        L = np.sqrt(dx**2 + dy**2)  # (1, n_det)
        ds = (L / (n_samples-1)).astype(np.float32)

        # --- 理想线积分（无噪声） ---
        line_int = (mu_samp.sum(axis=0) * ds).ravel()  # shape: (n_det,)

        if ADD_POISSON_NOISE:
            # Beer-Lambert: I = I0 * exp(-p)
            I_ideal = I0_PHOTONS * np.exp(-line_int.astype(np.float64))
            # Poisson 抽样
            Y = rng.poisson(I_ideal)
            # 避免 log(0)
            Y = np.maximum(Y.astype(np.float64), POISSON_CLIP_MIN)
            # 还原线积分：p_noisy = -ln(Y / I0)
            line_int = (-np.log(Y / float(I0_PHOTONS))).astype(np.float32)

        P[ib, :] = line_int.astype(np.float32)

    info = {
        "u_mm": u,
        "det_pitch_mm": det_pitch_mm,
        "beta_deg": beta_deg,
        "dso_mm": dso_mm,
        "dsd_mm": dsd_mm,
        "poisson": {
            "enabled": bool(ADD_POISSON_NOISE),
            "I0": float(I0_PHOTONS),
            "clip_min": float(POISSON_CLIP_MIN),
            "seed": None if RNG_SEED is None else int(RNG_SEED),
        }
    }
    return P, info



def fan2par_rebin(P: np.ndarray, info: Dict, n_t: int = PARALLEL_N_T, theta_deg_uniform: np.ndarray | None = None):
    """扇束->平行域重采样（theta = beta + gamma, t = dso*sin(gamma)）"""
    beta_deg = info["beta_deg"]
    u = info["u_mm"]
    dso = info["dso_mm"]
    dsd = info["dsd_mm"]

    beta = np.deg2rad(beta_deg)[:, None]
    gamma = np.arctan2(u, dsd)[None, :]

    theta = (beta + gamma)
    t = dso * np.sin(gamma)

    if theta_deg_uniform is None:
        th_min = np.rad2deg(theta.min())
        th_max = np.rad2deg(theta.max())
        ntheta = P.shape[0]
        theta_deg_uniform = np.linspace(th_min, th_max, ntheta)
    else:
        ntheta = len(theta_deg_uniform)

    if n_t is None:
        n_t = P.shape[1]

    TH_target = np.deg2rad(theta_deg_uniform)[:, None]
    t_min, t_max = float(t.min()), float(t.max())
    t_target = np.linspace(t_min, t_max, n_t)[None, :]

    P_par = np.zeros((ntheta, n_t), dtype=np.float32)

    theta_flat = theta
    # 先沿 theta 插值
    for j in range(len(u)):
        th_src = theta_flat[:, j]
        vals = P[:, j]
        # 二分查找；假定 th_src 单调
        idx = np.searchsorted(th_src, TH_target.ravel(), side='left')
        idx = np.clip(idx, 1, len(th_src)-1)
        th0 = th_src[idx-1]; th1 = th_src[idx]
        v0 = vals[idx-1];    v1 = vals[idx]
        w = (TH_target.ravel() - th0) / (th1 - th0 + EPS)
        vtheta = (1-w)*v0 + w*v1
        if j == 0:
            Vtheta_stack = np.zeros((len(u), len(vtheta)), dtype=np.float32)
        Vtheta_stack[j, :] = vtheta.astype(np.float32)

    # 再沿 t 插值
    t_src = t.ravel()
    order = np.argsort(t_src)
    t_src_sorted = t_src[order]
    Vtheta_sorted = Vtheta_stack[order, :]

    idx_t = np.searchsorted(t_src_sorted, t_target.ravel(), side='left')
    idx_t = np.clip(idx_t, 1, len(t_src_sorted)-1)
    t0 = t_src_sorted[idx_t-1]; t1 = t_src_sorted[idx_t]
    w_t = (t_target.ravel() - t0) / (t1 - t0 + EPS)
    for it in range(len(t_target.ravel())):
        v0 = Vtheta_sorted[idx_t[it]-1, :]
        v1 = Vtheta_sorted[idx_t[it], :]
        P_par[:, it] = ((1-w_t[it])*v0 + w_t[it]*v1)

    return P_par, {"theta_deg": theta_deg_uniform, "t_mm": t_target.ravel()}


def parallel_fbp(P_par: np.ndarray, theta_deg: np.ndarray, t_mm: np.ndarray,
                 out_size: Tuple[int, int], pixel_mm: float,
                 filter_name: str = FILTER_NAME, hann_cutoff: float = HANN_CUTOFF) -> np.ndarray:
    """平行束 FBP（频域滤波 + 反投影）。"""
    ntheta, nt = P_par.shape
    du = float(t_mm[1] - t_mm[0])

    if filter_name.lower() == "ram-lak":
        filt = ram_lak_filter(nt, du, hann_cutoff=None)
    elif filter_name.lower() == "hann":
        filt = ram_lak_filter(nt, du, hann_cutoff=hann_cutoff)
    elif filter_name.lower() == "shepp-logan":
        freqs = np.fft.fftfreq(nt, d=du)
        ram = np.abs(freqs) * 2.0
        sinc = np.sinc(freqs / (np.max(np.abs(freqs))+EPS))
        filt = ram * np.abs(sinc)
    else:
        # 默认：Hann
        filt = ram_lak_filter(nt, du, hann_cutoff=0.8)

    P_f = np.zeros_like(P_par, dtype=np.float32)
    for i in range(ntheta):
        F = np.fft.fft(P_par[i, :])
        P_f[i, :] = np.fft.ifft(F * filt).real.astype(np.float32)

    H, W = out_size
    cx = (W - 1) / 2.0
    cy = (H - 1) / 2.0
    xs = (np.arange(W) - cx) * pixel_mm
    ys = (np.arange(H) - cy) * pixel_mm
    X, Y = np.meshgrid(xs, ys)

    recon = np.zeros((H, W), dtype=np.float32)
    for i, th in enumerate(np.deg2rad(theta_deg)):
        t_xy = X * np.cos(th) + Y * np.sin(th)
        idx = (t_xy - t_mm[0]) / (t_mm[1] - t_mm[0])
        idx0 = np.floor(idx).astype(int)
        idx1 = idx0 + 1
        w = (idx - idx0).astype(np.float32)

        idx0 = np.clip(idx0, 0, len(t_mm)-1)
        idx1 = np.clip(idx1, 0, len(t_mm)-1)

        v0 = P_f[i, idx0]
        v1 = P_f[i, idx1]
        val = (1.0 - w) * v0 + w * v1
        recon += val.astype(np.float32)

    if len(theta_deg) > 1:
        delta_theta = np.deg2rad(np.mean(np.diff(theta_deg)))
    else:
        delta_theta = 0.0
    recon *= delta_theta

    return recon



In [5]:
# =========================================================
# 角度采样（按步长），以及（a）按角度阈值切分并对比到（b）/（c）
# =========================================================
def arange_with_end(start: float, end: float, step: float, include_end: bool = True) -> np.ndarray:
    """
    像 np.arange，但可确保包含终点（若整除或接近时）。
    """
    arr = np.arange(start, end + (step if include_end else 0.0) + 1e-7, step)
    if include_end:
        if arr[-1] > end + 1e-6:
            arr = arr[:-1]
        if abs(arr[-1] - end) > 1e-6:
            arr = np.concatenate([arr, np.array([end])])
    return arr.astype(float)


def build_angle_sets(step_deg: float, include_endpoints: bool = True):
    """
    生成四组角度：
      (a) 0–180（含端点），按步长采样
      (b) 0–90  同样步长采样
      (c) 90–180 同样步长采样
      (d) 从 (a) 里**按角度阈值**严格切出两半：0–90、90–180
    """
    a = arange_with_end(0.0, 180.0, step_deg, include_endpoints)
    b = arange_with_end(0.0, 90.0, step_deg, include_endpoints)
    c = arange_with_end(90.0, 180.0, step_deg, include_endpoints)

    # —— 按角度阈值切分 (a) ——（严格 <=90 与 >=90）
    a_first = a[(a >= 0.0 - 1e-6) & (a <= 90.0 + 1e-6)]
    a_second = a[(a >= 90.0 - 1e-6) & (a <= 180.0 + 1e-6)]

    return {
        "a_0_180": a,
        "b_0_90": b,
        "c_90_180": c,
        "a_first_from_a": a_first,     # 用于与 b_0_90 对比
        "a_second_from_a": a_second,   # 用于与 c_90_180 对比
    }


In [6]:
# =========================================================
# 主流程
# =========================================================
# 1) 读取 DICOM 并转换为 HU / mu
ds = pydicom.dcmread(str(DCM_PATH))
img_raw = ds.pixel_array.astype(np.float32)

# 像素间距
pixel_spacing = getattr(ds, "PixelSpacing", [PIXEL_MM_FALLBACK, PIXEL_MM_FALLBACK])
PIXEL_MM = float(pixel_spacing[0])  # 假设方形像素

# HU 转换
slope = float(getattr(ds, "RescaleSlope", 1.0))
intercept = float(getattr(ds, "RescaleIntercept", 0.0))
img_hu = img_raw * slope + intercept

# 选择底图
if BASE_IMAGE_TYPE.upper() == "HU":
    BASE_IMG = img_hu.astype(np.float32)
elif BASE_IMAGE_TYPE.upper() == "MU":
    BASE_IMG = hu_to_mu(img_hu).astype(np.float32)
else:
    raise ValueError("BASE_IMAGE_TYPE must be 'HU' or 'MU'")

IMG_SIZE = BASE_IMG.shape  # (H, W)

# 保存原图（用于记录）
plt.figure(); plt.imshow(img_hu, cmap='gray'); plt.title("HU (from DICOM)"); plt.axis('off')
plt.savefig(OUTDIR / "input_HU.png", bbox_inches='tight'); plt.close()

if BASE_IMAGE_TYPE.upper() == "MU":
    plt.figure(); plt.imshow(BASE_IMG, cmap='gray'); plt.title("mu (mm^-1) from HU"); plt.axis('off')
    plt.savefig(OUTDIR / "input_mu.png", bbox_inches='tight'); plt.close()


In [7]:
# 2) Ground Truth（以 BASE_IMG 作为 “GT”）
GT = BASE_IMG.copy()
gt_n = to_float01(GT)



In [8]:
# 3) 统一计算 -> 汇总 -> 统一绘图
all_rows = []    # 记录主三组(0_180,0_90,90_180)的 step-SSIM
pair_rows = []   # 记录配对对比（a_first_from_a vs b_0_90；a_second_from_a vs c_90_180）的指标

# ====== Compute 阶段：逐 step 先把投影/重建全部跑完并缓存 ======
# data_cache[step] = {
#   "recon": {"0_180": img, "0_90": img, "90_180": img, "a_first": img, "a_second": img},
#   "sinogram": {"0_180": P, ...},
#   "ssim": {"0_180": val, ...}  # 只记录三主组的 SSIM；a_first/a_second 的 SSIM在pair时另算
# }
data_cache: dict[float, dict] = {}

for step in ANGLE_STEP_LIST:
    step_dir = OUTDIR / f"step_{int(step)}"
    recons_dir = step_dir / "recons"
    sinos_dir = step_dir / "sinos"
    panels_dir = step_dir / "panels"
    compare_dir = step_dir / "pairwise_compare"
    ssimmap_dir = compare_dir / "ssim_maps_vs_GT"  # 存放对比里两幅图相对GT的SSIM热图
    for d in [recons_dir, sinos_dir, panels_dir, compare_dir, ssimmap_dir]:
        os.makedirs(d, exist_ok=True)

    angle_sets = build_angle_sets(step, INCLUDE_ENDPOINTS)

    cache_recon = {}
    cache_sino = {}
    cache_ssim = {}

    # 三主组（a,b,c）
    eval_groups = [
        ("a_0_180", "0_180"),
        ("b_0_90", "0_90"),
        ("c_90_180", "90_180"),
    ]

    for key, label in eval_groups:
        beta = angle_sets[key]
        sino_fan, info = fanbeam_forward(BASE_IMG, beta, pixel_mm=PIXEL_MM, dso_mm=DSO, dsd_mm=DSD)

        # 存 sinogram（按 step 子目录）
        plt.figure()
        plt.imshow(
            sino_fan, cmap='gray', aspect='auto', origin='lower',
            extent=[0, sino_fan.shape[1]-1, float(beta.min()), float(beta.max())]
        )
        plt.xlabel("detector index")
        plt.ylabel("angle (deg)")
        plt.title(f"Fan-beam sinogram {label} (step={step}°)")
        plt.savefig(sinos_dir / f"sino_{label}.png", bbox_inches='tight')
        plt.close()

        P_par, par_info = fan2par_rebin(sino_fan, info, n_t=PARALLEL_N_T)
        recon = parallel_fbp(P_par, par_info["theta_deg"], par_info["t_mm"],
                             out_size=IMG_SIZE, pixel_mm=PIXEL_MM,
                             filter_name=FILTER_NAME, hann_cutoff=HANN_CUTOFF)
        recon_aligned = rot90_cw(recon)

        cache_recon[label] = recon_aligned
        cache_sino[label] = sino_fan

        # 保存单图（按 step 子目录）
        if SAVE_PER_RECON:
            plt.figure()
            plt.imshow(recon_aligned, cmap="gray")
            plt.axis("off")
            plt.title(f"{label} | step={step}°")
            plt.savefig(recons_dir / f"recon_{label}.png", bbox_inches="tight", pad_inches=0.0)
            plt.close()

        # SSIM（对 GT）
        score = ssim_circle(gt_n, to_float01(recon_aligned)) if SSIM_USE_CENTER_CIRCLE else ssim_roi_box(GT, recon_aligned)
        cache_ssim[label] = float(score)
        all_rows.append({"step_deg": float(step), "range": label, "SSIM": float(score)})

    # (d) 从 (a) 严格按角度阈值切出两半
    # 前半：0–90
    beta_afirst = angle_sets["a_first_from_a"]
    if len(beta_afirst) >= 2:
        sino_fan_h1, info_h1 = fanbeam_forward(BASE_IMG, beta_afirst, pixel_mm=PIXEL_MM, dso_mm=DSO, dsd_mm=DSD)
        P_par_h1, par_info_h1 = fan2par_rebin(sino_fan_h1, info_h1, n_t=PARALLEL_N_T)
        recon_h1 = parallel_fbp(P_par_h1, par_info_h1["theta_deg"], par_info_h1["t_mm"],
                                out_size=IMG_SIZE, pixel_mm=PIXEL_MM,
                                filter_name=FILTER_NAME, hann_cutoff=HANN_CUTOFF)
        cache_recon["a_first"] = rot90_cw(recon_h1)
        cache_sino["a_first"]  = sino_fan_h1

    # 后半：90–180
    beta_asecond = angle_sets["a_second_from_a"]
    if len(beta_asecond) >= 2:
        sino_fan_h2, info_h2 = fanbeam_forward(BASE_IMG, beta_asecond, pixel_mm=PIXEL_MM, dso_mm=DSO, dsd_mm=DSD)
        P_par_h2, par_info_h2 = fan2par_rebin(sino_fan_h2, info_h2, n_t=PARALLEL_N_T)
        recon_h2 = parallel_fbp(P_par_h2, par_info_h2["theta_deg"], par_info_h2["t_mm"],
                                out_size=IMG_SIZE, pixel_mm=PIXEL_MM,
                                filter_name=FILTER_NAME, hann_cutoff=HANN_CUTOFF)
        cache_recon["a_second"] = rot90_cw(recon_h2)
        cache_sino["a_second"]  = sino_fan_h2

    # 暂存到总 cache
    data_cache[step] = {"recon": cache_recon, "sinogram": cache_sino, "ssim": cache_ssim}

In [9]:
# ====== Aggregate 阶段：统计表 ======
df = pd.DataFrame(all_rows).sort_values(["step_deg", "range"])
if SAVE_CSV:
    csv_path = OUTDIR / "ssim_vs_step.csv"
    df.to_csv(csv_path, index=False)
    print(f"[Info] Saved CSV -> {csv_path}")

[Info] Saved CSV -> /home/xm/project1/output/limited_angle_proj_test/ssim_vs_step.csv


In [10]:
# ====== Plot 阶段：统一绘图 ======

def display_name(key: str) -> str:
    """将内部键转换为图上更易读的标题，不改变文件名。"""
    mapping = {
        "0_90": "0-90",
        "90_180": "90-180",
        "0_180": "0-180",
        "a_first": "0-90 from 0-180",
        "a_second": "90-180 from 0-180",
        # 兼容旧键名（若缓存里有）
        "a_first(0_90 from a)": "0-90 from 0-180",
        "a_second(90_180 from a)": "90-180 from 0-180",
        "a_first(≈0_90)": "0-90 from 0-180",
        "a_second(≈90_180)": "90-180 from 0-180",
    }
    return mapping.get(key, key)

# 1) “不同角度范围”的 SSIM 柱状图（取最小步长）
if len(ANGLE_STEP_LIST) > 0 and len(df) > 0:
    step_demo = min(ANGLE_STEP_LIST)
    df_demo = df[df["step_deg"] == step_demo]
    range_order = ["0_90", "90_180", "0_180"]
    vals = [float(df_demo[df_demo["range"] == r]["SSIM"].mean()) if (df_demo["range"] == r).any() else np.nan
            for r in range_order]
    plt.figure(figsize=(6, 3))
    bars = plt.bar([display_name(r) for r in range_order], vals)
    plt.ylabel("SSIM vs GT")
    plt.xlabel("Angle Range")
    plt.ylim(0, 1)
    plt.grid(axis='y', alpha=0.3)
    plt.title(f"SSIM vs GT (step={step_demo}°)")
    for bar, val in zip(bars, vals):
        if np.isfinite(val):
            plt.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.3f}",
                     ha='center', va='bottom', fontsize=8)
    plt.savefig(OUTDIR / f"ssim_bar_step{int(step_demo)}.png", bbox_inches='tight')
    plt.close()

# 2) “SSIM vs 步长”的曲线（分别针对 0_90、90_180、0_180）
if len(df) > 0:
    plt.figure(figsize=(7, 3.5))
    for r in ["0_90", "90_180", "0_180"]:
        dfr = df[df["step_deg"] == df["step_deg"]][df["range"] == r].sort_values("step_deg")  # 保守写法
        dfr = df[df["range"] == r].sort_values("step_deg")
        if len(dfr) == 0:
            continue
        plt.plot(dfr["step_deg"], dfr["SSIM"], marker="o", label=display_name(r))
    plt.xlabel("Angle step (deg)")
    plt.ylabel("SSIM vs GT")
    margin = 0.05
    ymin = max(0.0, np.floor((df["SSIM"].min() - margin) * 10) / 10.0)
    ymax = min(1.0, np.ceil((df["SSIM"].max() + margin) * 10) / 10.0)
    plt.ylim(ymin, ymax)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.title("SSIM vs Angle Step for Angle Ranges")
    plt.tight_layout()
    plt.savefig(OUTDIR / "ssim_vs_step.png", bbox_inches="tight")
    plt.close()

# 3) 每个 step 出一张“面板图”（可选）
if SAVE_PANEL:
    for step, pack in data_cache.items():
        step_dir = OUTDIR / f"step_{int(step)}"
        panels_dir = step_dir / "panels"
        recons = pack["recon"]; ssim_dict = pack["ssim"]
        keys_in_order = ["0_180", "0_90", "90_180", "a_first", "a_second"]
        imgs = [(k, recons[k]) for k in keys_in_order if k in recons]
        if len(imgs) == 0:
            continue
        fig, axes = plt.subplots(1, len(imgs), figsize=(3*len(imgs), 3), constrained_layout=True)
        if len(imgs) == 1:
            axes = [axes]
        for ax, (k, im) in zip(axes, imgs):
            ax.imshow(to_uint8(im), cmap="gray", vmin=0, vmax=255)
            ax.axis("off")
            title_ssim = display_name(k)
            if k in ["0_180", "0_90", "90_180"] and k in ssim_dict:
                title_ssim += f"\nSSIM={ssim_dict[k]:.3f}"
            ax.set_title(title_ssim)
        fig.suptitle(f"FBP (step={step}°, filter={FILTER_NAME})")
        fig.savefig(panels_dir / f"panel_step{int(step)}.png", bbox_inches="tight")
        plt.close(fig)

# 4) 成对对比（a 的两半 vs b/c）：同一张图内展示两幅重建 + 差值灰度图，并显示 SSIM(A,B)
def ssim_value(img):
    return (ssim_circle(gt_n, to_float01(img)) if SSIM_USE_CENTER_CIRCLE else ssim_roi_box(GT, img))

def ssim_pair(a_img, b_img) -> float:
    # 两图互相之间的 SSIM（基于 0-1 归一化）
    val, _ = ssim(to_float01(a_img), to_float01(b_img), data_range=1.0, full=True)
    return float(val)

def save_ssim_map_vs_gt(img, save_path, title_prefix=""):
    score, ssim_map = ssim(gt_n, to_float01(img), data_range=1.0, full=True)
    plt.figure()
    plt.imshow(ssim_map, cmap="viridis")
    plt.colorbar()
    prefix = f"{title_prefix} " if title_prefix else ""
    plt.title(f"{prefix}SSIM map vs GT (mean={score:.3f})")
    plt.axis("off")
    plt.savefig(save_path, bbox_inches="tight")
    plt.close()

for step, pack in data_cache.items():
    step_dir = OUTDIR / f"step_{int(step)}"
    compare_dir = step_dir / "pairwise_compare"
    ssimmap_dir = compare_dir / "ssim_maps_vs_GT"
    os.makedirs(compare_dir, exist_ok=True)
    os.makedirs(ssimmap_dir, exist_ok=True)

    recons = pack["recon"]

    # a_first vs b_0_90
    if "a_first" in recons and "0_90" in recons:
        A = recons["a_first"]; B = recons["0_90"]
        ssim_A_gt = ssim_value(A); ssim_B_gt = ssim_value(B)
        ssim_AB = ssim_pair(A, B)  # 新增：A 与 B 的 SSIM

        # 记录表（保留原字段名）
        pair_rows.append({
            "step_deg": float(step),
            "pair": "a_first_vs_b_0_90",
            "SSIM_a_first": float(ssim_A_gt),
            "SSIM_b_0_90": float(ssim_B_gt),
            "SSIM_A_B": float(ssim_AB),   # 新增字段
        })

        # 中间灰度差值图：对称归一化到 [0,1]（0.5 ≈ 零差，黑<0，白>0）
        d = to_float01(A) - to_float01(B)
        vmax = max(1e-8, np.max(np.abs(d)))
        diff_gray = (d + vmax) / (2.0 * vmax)  # [-vmax, vmax] -> [0,1]

        fig, axes = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
        axes[0].imshow(to_uint8(A), cmap="gray", vmin=0, vmax=255)
        axes[0].set_title(f"{display_name('a_first')} (SSIM vs GT={ssim_A_gt:.3f})"); axes[0].axis("off")

        axes[1].imshow(diff_gray, cmap="gray", vmin=0, vmax=1)
        axes[1].set_title("A - B (grayscale)"); axes[1].axis("off")

        axes[2].imshow(to_uint8(B), cmap="gray", vmin=0, vmax=255)
        axes[2].set_title(f"{display_name('0_90')} (SSIM vs GT={ssim_B_gt:.3f})"); axes[2].axis("off")

        fig.suptitle(f"Pairwise Compare (step={step}°)  |  SSIM(A, B)={ssim_AB:.3f}")
        fig.savefig(compare_dir / f"pair_aFirst_vs_b_{int(step)}.png", bbox_inches="tight")
        plt.close(fig)

        # 两张图各自“SSIM map vs GT”（保持不变）
        save_ssim_map_vs_gt(A, ssimmap_dir / f"ssim_map_aFirst_vs_GT_step{int(step)}.png",
                            title_prefix=display_name('a_first'))
        save_ssim_map_vs_gt(B, ssimmap_dir / f"ssim_map_b_0_90_vs_GT_step{int(step)}.png",
                            title_prefix=display_name('0_90'))

    # a_second vs c_90_180
    if "a_second" in recons and "90_180" in recons:
        A2 = recons["a_second"]; C = recons["90_180"]
        ssim_A2_gt = ssim_value(A2); ssim_C_gt = ssim_value(C)
        ssim_A2C = ssim_pair(A2, C)  # 新增：A2 与 C 的 SSIM

        pair_rows.append({
            "step_deg": float(step),
            "pair": "a_second_vs_c_90_180",
            "SSIM_a_second": float(ssim_A2_gt),
            "SSIM_c_90_180": float(ssim_C_gt),
            "SSIM_A_B": float(ssim_A2C),  # 复用字段名以对齐，也可以改成 SSIM_A_C
        })

        d2 = to_float01(A2) - to_float01(C)
        vmax2 = max(1e-8, np.max(np.abs(d2)))
        diff_gray2 = (d2 + vmax2) / (2.0 * vmax2)

        fig2, axes2 = plt.subplots(1, 3, figsize=(9, 3), constrained_layout=True)
        axes2[0].imshow(to_uint8(A2), cmap="gray", vmin=0, vmax=255)
        axes2[0].set_title(f"{display_name('a_second')} (SSIM vs GT={ssim_A2_gt:.3f})"); axes2[0].axis("off")

        axes2[1].imshow(diff_gray2, cmap="gray", vmin=0, vmax=1)
        axes2[1].set_title("A - B (grayscale)"); axes2[1].axis("off")

        axes2[2].imshow(to_uint8(C), cmap="gray", vmin=0, vmax=255)
        axes2[2].set_title(f"{display_name('90_180')} (SSIM vs GT={ssim_C_gt:.3f})"); axes2[2].axis("off")

        fig2.suptitle(f"Pairwise Compare (step={step}°)  |  SSIM(A, B)={ssim_A2C:.3f}")
        fig2.savefig(compare_dir / f"pair_aSecond_vs_c_{int(step)}.png", bbox_inches="tight")
        plt.close(fig2)

        save_ssim_map_vs_gt(A2, ssimmap_dir / f"ssim_map_aSecond_vs_GT_step{int(step)}.png",
                            title_prefix=display_name('a_second'))
        save_ssim_map_vs_gt(C,  ssimmap_dir / f"ssim_map_c_90_180_vs_GT_step{int(step)}.png",
                            title_prefix=display_name('90_180'))


# 5) 成对对比的汇总 CSV（在所有绘图之后统一落盘）
df_pair = pd.DataFrame(pair_rows).sort_values(["step_deg", "pair"])
if SAVE_CSV and len(df_pair) > 0:
    pair_csv = OUTDIR / "pairwise_compare_summary.csv"
    df_pair.to_csv(pair_csv, index=False)
    print(f"[Info] Saved pairwise comparison CSV -> {pair_csv}")

print(f"[Done] All outputs saved under: {OUTDIR}")


[Info] Saved pairwise comparison CSV -> /home/xm/project1/output/limited_angle_proj_test/pairwise_compare_summary.csv
[Done] All outputs saved under: /home/xm/project1/output/limited_angle_proj_test
